# 03 — Rules baseline: the evidence

Regenerates every lift table quoted in [`docs/rules-baseline.md`](../docs/rules-baseline.md).

**This notebook is not the record.** `rules-baseline.md` is; the weights it justifies
are committed in `config/config.yaml`. This is the *reproduction path* — the thing that
was missing when the rules engine shipped, and the reason a reader had to take the
tables on trust.

**Why a separate notebook from `01_eda.ipynb`.** Each rule carries a `provenance` field:
`hypothesis` for the ones formed in Phase 01 before any modelling, `search` for the ones
found here by scanning a column family. That distinction is what prices the selection
bias in R4 and R5, and it is only credible because the two happened in separate
artifacts. Folding the `D*`/`M*` scan into the EDA notebook would make the search look
like it happened before the modelling it informed.

**Evidence window: TRAIN only, days 1–90.** Narrower than the EDA horizon of 120. The
incumbent must never be fitted, tuned or checked against validation — an incumbent
selected on the challenger's data makes the Phase 06 comparison meaningless.

> ⚠️ **If a number here disagrees with `rules-baseline.md`, that is a finding to write
> down — not a reason to edit `config.yaml` to match.** The weights are committed. The
> half-split check is what set R5 to 1 instead of 3; if those numbers move, the
> discrepancy is the interesting artifact.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from fraud_engine.data.load import DEFAULT_CONFIG_PATH, load_config
from fraud_engine.data.splits import resolve_boundaries

In [ ]:
# A notebook's cwd is unreliable - anchor to the repo root instead.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

config = load_config(ROOT / DEFAULT_CONFIG_PATH)
interim_path = ROOT / config["paths"]["interim"]
rules_cfg = config["baselines"]["rules"]

In [ ]:
# Derived from the split config, never written out as a literal: `train_end` is
# `val_fit_start - gap_days - 1`, so a change to the gap moves this window with it.
# A hardcoded 90 here would silently keep the old window after an E1 ablation.
TRAIN_FIRST, TRAIN_LAST = resolve_boundaries(config["splits"])["train"]

# Half A / half B, for the stability check. The midpoint is derived too.
HALF_SPLIT_DAY = TRAIN_FIRST + (TRAIN_LAST - TRAIN_FIRST) // 2

TRAIN_FILTER = [("day", ">=", TRAIN_FIRST), ("day", "<=", TRAIN_LAST)]


# One place enforces the window - a bare read_parquet later would span all 182 days
# and quietly measure the rules against data they must never see.
def read_train(columns):
    return pd.read_parquet(interim_path, columns=columns, filters=TRAIN_FILTER)


print(f"train days {TRAIN_FIRST}-{TRAIN_LAST}, halves split at day {HALF_SPLIT_DAY}")

### Section 1 — the lift helper and the weighting rule

Everything below is one measurement repeated: what share of the rows a predicate
selects, and how much richer in fraud they are than the population.

```
lift = fraud_rate(fires) / fraud_rate(all)
weight = round( min(lift_half_A, lift_half_B) )
```

The **conservative half**, not the average and not the full-train figure. Stated in
advance and computed on train, it discounts an unstable rule automatically rather than
by judgement — which is the whole defence against the selection bias in the `search`
rules.

Integer points, not real-valued weights: a baseline that emits `lift`-valued scores is a
small fitted model wearing an incumbent's name. The rounding is also what keeps an
unstable 1.16× and a stable 1.42× from being treated as meaningfully different.

In [ ]:
# Fraud rate and lift for the rows a predicate selects.
#
# `base` is explicit because "lift" is a slippery word: the same predicate measured
# against a subset's base rate and against the whole window's gives two different
# numbers, and neither is wrong - they answer different questions. A CONTROL asks
# "does the effect survive within product", so it lifts against the subset. A rule
# that will be RANKED against all transactions must lift against the whole window,
# because that is the population it competes in.
def lift(frame, fires, label="", base=None):
    fires = fires.fillna(False).astype(bool)
    base = frame["isFraud"].mean() if base is None else base
    selected = frame.loc[fires]
    days = frame["day"].nunique()

    return {
        "rule": label,
        "rows": int(fires.sum()),
        "rows_per_day": round(fires.sum() / days, 1),
        "fraud_rate": selected["isFraud"].mean() if len(selected) else np.nan,
        "lift": (selected["isFraud"].mean() / base) if len(selected) else np.nan,
        "fraud_usd": round(selected.loc[selected["isFraud"] == 1, "TransactionAmt"].sum(), 0),
    }


# The same lift measured in each half of train, and the weight that implies.
#
# The halves are the correction for selection bias: a rule found by scanning twenty
# combinations has an optimistic full-window figure by construction, and only a split
# it was not selected on can price that.
def stability(frame, fires, label=""):
    fires = fires.fillna(False).astype(bool)
    halves = {}
    for name, mask in (
        ("A", frame["day"] < HALF_SPLIT_DAY),
        ("B", frame["day"] >= HALF_SPLIT_DAY),
    ):
        half = frame.loc[mask]
        halves[f"lift_{name}"] = lift(half, fires.loc[mask])["lift"]

    weakest = min(halves["lift_A"], halves["lift_B"])
    return {
        "rule": label,
        **halves,
        "lift_full": lift(frame, fires)["lift"],
        "weakest_half": round(weakest, 2),
        "implied_weight": round(weakest),
    }


def show(rows):
    return pd.DataFrame(rows).set_index("rule").round(3)

### Section 2 — R1: round amount in the $150–$500 band

`provenance: hypothesis` — H2, formed in Phase 01 and written down before any modelling.

The band is asserted from H2 and lives in config; only its continued validity is
re-checked here. Nothing is fitted.

In [ ]:
r1 = read_train(["day", "isFraud", "TransactionAmt", "ProductCD"])

cfg = rules_cfg["round_amount"]
r1_fires = r1["TransactionAmt"].mod(cfg["step"]).eq(0) & r1["TransactionAmt"].between(
    cfg["min"], cfg["max"]
)

display(show([lift(r1, r1_fires, "R1 round_amount")]))
display(show([stability(r1, r1_fires, "R1 round_amount")]))

In [ ]:
# The band is specific, not a general preference for round numbers. $100 is the most
# common round amount in the data and runs BACKWARDS; everything above $500 is at parity.
base = r1["isFraud"].mean()
rows = []
for amount in (50, 100, 150, 200, 250, 300, 350, 400, 450, 500, 550, 600):
    rows.append(lift(r1, r1["TransactionAmt"].eq(amount), f"${amount}"))
show(rows)

In [ ]:
# H2 survived the ProductCD control and STRENGTHENED under it - the reason R1 kept its
# slot. The within-product rows lift against their own product's base rate: that is what
# a control means, and the pooled figure is diluted across products.
#
# These are measured HERE on train (days 1-90). hypotheses.md quotes the same comparison
# on the Phase 01 horizon (days 1-120) and its figures are larger. The DIRECTION is what
# H2 claimed and it reproduces; the magnitudes do not, and Section 9 records that.
rows = []
for amount in (300, 450):
    rows.append(lift(r1, r1["TransactionAmt"].eq(amount), f"${amount} pooled"))
    for product in ("H", "R"):
        within = r1[r1["ProductCD"].astype(str) == product]
        entry = lift(within, within["TransactionAmt"].eq(amount), f"${amount} within {product}")
        rows.append(entry)
show(rows)

In [ ]:
# Volume is unstable, lift is not: R1 fires far less often in half B. That is not decay
# - it tracks the product-mix shift recorded under Limitations. What remains is more
# concentrated, which is why the lift RISES as the volume falls.
for name, mask in (("A", r1["day"] < HALF_SPLIT_DAY), ("B", r1["day"] >= HALF_SPLIT_DAY)):
    half = r1.loc[mask]
    entry = lift(half, r1_fires.loc[mask], f"R1 half {name}")
    print(f"half {name}: {entry['rows_per_day']:>6} rows/day   lift {entry['lift']:.2f}x")

### Section 3 — R2: product tier

`provenance: hypothesis` — H3.

Tiered rather than binary so one rule slot carries the entire channel axis. The tier
map is a judgement recorded in config, derived from the rates below.

In [ ]:
r2 = read_train(["day", "isFraud", "TransactionAmt", "ProductCD"])
days = r2["day"].nunique()

tiers = rules_cfg["product_tier"]
product_table = (
    r2.assign(product=r2["ProductCD"].astype(str))
    .groupby("product")
    .apply(
        lambda g: pd.Series(
            {
                "rows_per_day": round(len(g) / days, 1),
                "fraud_rate": g["isFraud"].mean(),
                "lift": g["isFraud"].mean() / r2["isFraud"].mean(),
                "fraud_usd": round(g.loc[g["isFraud"] == 1, "TransactionAmt"].sum(), 0),
            }
        ),
        include_groups=False,
    )
)
product_table["tier"] = pd.Series(tiers)
display(product_table.round(3))

In [ ]:
# R and W score zero because both sit BELOW the base rate. That is "nothing added",
# not evidence of safety - the distinction matters when reading a score of 0.
below = product_table[product_table["lift"] < 1.0].index.tolist()
print(f"below base rate, so tier 0: {below}")

# C's stability, the figure the tier rests on.
c_fires = r2["ProductCD"].astype(str).eq("C")
display(show([stability(r2, c_fires, "R2 ProductCD == C")]))

In [ ]:
# The money runs the other way, and that is the point. A policy tuned on RATE alone
# chases C and misses where the money is. The rules engine has no per-transaction cost
# model and cannot resolve this - a structural reason the Phase 05 model should beat it.
display(
    product_table[["fraud_rate", "fraud_usd"]]
    .sort_values("fraud_rate", ascending=False)
    .assign(usd_rank=lambda d: d["fraud_usd"].rank(ascending=False).astype(int))
    .round(4)
)

### Section 4 — R3: amount above the within-product 99th percentile

`provenance: hypothesis` — H1, in the form that survived.

H1 was falsified pooled and holds only *within* product, so the threshold is
per-product by construction. A global cut would be the pooled artifact H1 was revised
to reject. **The five cut points are the only fitted values in the engine.**

In [ ]:
r3 = read_train(["day", "isFraud", "TransactionAmt", "ProductCD"])
product = r3["ProductCD"].astype(str)

quantile = rules_cfg["amount_percentile"]["quantile"]
cuts = r3.groupby(product, observed=True)["TransactionAmt"].quantile(quantile)
print(f"fitted cut points at q={quantile}: {cuts.round(2).to_dict()}")

r3_fires = r3["TransactionAmt"] > product.map(cuts).astype("float64")
display(show([lift(r3, r3_fires, "R3 amount_percentile")]))
display(show([stability(r3, r3_fires, "R3 amount_percentile")]))

In [ ]:
# The percentile LEVEL was chosen for precision per review slot, not for lift: p90 and
# p95 fire many times more often at essentially the same lift, and the extra volume buys
# nothing against a ~35/day review budget.
rows = []
for q in (0.90, 0.95, 0.99):
    q_cuts = r3.groupby(product, observed=True)["TransactionAmt"].quantile(q)
    fires = r3["TransactionAmt"] > product.map(q_cuts).astype("float64")
    rows.append(lift(r3, fires, f"within-product p{int(q * 100)}"))
show(rows)

### Section 5 — R4: new-card proxy

`provenance: **search**` — a directed scan of the `D*` family on train.

Selection-biased by construction: roughly twenty flag/value combinations were scanned
across `D*` and `M*` and the best kept, so the full-train lift is optimistic by an
unmeasured amount. The half-split check is the correction.

In [ ]:
r4 = read_train(["day", "isFraud", "TransactionAmt", "D1"])

# The cleanest monotone signal in the data - and the reason the cut is a choice from
# this table rather than a fitted threshold.
bands = [(0, 0), (1, 1), (2, 3), (4, 7), (8, 14), (15, 30), (31, 90), (91, 180), (181, 10_000)]
rows = [lift(r4, r4["D1"].between(lo, hi), f"D1 {lo}-{hi}") for lo, hi in bands]
show(rows)

In [ ]:
# D1 == 0 is excluded DELIBERATELY: it covers about half of all rows at barely above
# base rate, so including it would convert a precise rule into a broad one.
zero_share = r4["D1"].eq(0).mean()
print(f"D1 == 0 covers {zero_share:.1%} of train rows")

r4_fires = r4["D1"].gt(0) & r4["D1"].le(rules_cfg["new_card"]["d1_max"])
display(show([lift(r4, r4_fires, "R4 new_card")]))
display(show([stability(r4, r4_fires, "R4 new_card")]))

### Section 6 — R5: product W with `M4 == "M2"`

`provenance: **search**` — a directed scan of the `M*` family.

**This is the rule the selection caveat is about.** The half-split check caught it and
the weight of 1 is that correction. It is also the only rule off the amount/product
axis, which is why it earns a slot despite a modest weight.

In [ ]:
r5 = read_train(["day", "isFraud", "TransactionAmt", "ProductCD", "M4"])
product = r5["ProductCD"].astype(str)
m4 = r5["M4"].astype("object")

# The ProductCD == "W" scope is load-bearing, not decoration. Pooled, M4 == "M2" looks
# strong - but almost all of its rows are product C, where its lift is ~1. Unscoped the
# rule is R2 restated for the third time.
rows = [lift(r5, m4.eq("M2"), "M4 == M2 pooled")]
print(f"share of M4 == M2 rows that are product C: {product[m4.eq('M2')].eq('C').mean():.1%}")
for scope in ("C", "W"):
    within = r5[product == scope]
    rows.append(lift(within, within["M4"].astype("object").eq("M2"), f"M4 == M2 within {scope}"))
show(rows)

In [ ]:
r5_fires = product.eq("W") & m4.eq("M2")
display(show([lift(r5, r5_fires, "R5 w_m4_m2")]))
display(show([stability(r5, r5_fires, "R5 w_m4_m2")]))

### Section 7 — the rejected rules

Three of these are suggested by the ROADMAP itself. All were dropped on measured
evidence, not preference — which is the part worth reproducing, because "we tried it
and it was backwards" is a stronger claim than "we didn't use it".

In [ ]:
# Email-domain mismatch - falsified, and BACKWARDS. On the rows where both domains are
# present the direction is the opposite of the premise.
rej = read_train(
    ["day", "isFraud", "TransactionAmt", "ProductCD", "P_emaildomain", "R_emaildomain"]
)
print(f"R_emaildomain null on {rej['R_emaildomain'].isna().mean():.1%} of train")

# ...and the nullness is structural rather than informative.
null_by_product = rej.groupby(rej["ProductCD"].astype(str))["R_emaildomain"].apply(
    lambda s: s.isna().mean()
)
display(null_by_product.round(3).rename("R_emaildomain null share"))

# Lifted against the TRAIN base rate, not against the both-present subset's. The rule
# would be scored alongside every other transaction, so the whole window is the
# population it competes in - the same argument that sinks `M2 == "F"` two cells below.
# Against the subset's own base rate these read 1.18x and 0.32x, and the reversal is
# just as clear; the pooled figures are the decision-relevant ones.
train_base = rej["isFraud"].mean()

both = rej[rej["P_emaildomain"].notna() & rej["R_emaildomain"].notna()]
matched = both["P_emaildomain"].astype(str) == both["R_emaildomain"].astype(str)
display(
    show(
        [
            lift(both, matched, "domains match", base=train_base),
            lift(both, ~matched, "domains mismatch", base=train_base),
        ]
    )
)

# As specified the rule fires on a twentieth of train and catches less fraud than that
# share - worse than selecting rows at random.
fires_share = (~matched).sum() / len(rej)
caught_share = both.loc[~matched, "isFraud"].sum() / rej["isFraud"].sum()
print(f"fires on {fires_share:.2%} of train, catches {caught_share:.1%} of its fraud")

In [ ]:
# Missing identity - direction reverses once controlled for product. Pooled it looks
# like a strong signal; within every product that admits a comparison it points the
# other way, and W is 100% no-identity so it admits none at all.
ident = read_train(["day", "isFraud", "TransactionAmt", "ProductCD", "has_identity"])
no_id = ~ident["has_identity"].astype(bool)

rows = [lift(ident, no_id, "no identity, pooled")]
for scope in ("C", "H", "R", "S", "W"):
    within = ident[ident["ProductCD"].astype(str) == scope]
    rows.append(lift(within, ~within["has_identity"].astype(bool), f"no identity within {scope}"))
show(rows)

In [ ]:
# ...and the inverted rule is real but REDUNDANT: it agrees with `ProductCD != W` on
# almost every row, making it a third pass at the channel axis R2 already covers.
agreement = (ident["has_identity"].astype(bool) == ident["ProductCD"].astype(str).ne("W")).mean()
print(f"has_identity agrees with ProductCD != W on {agreement:.1%} of train rows")

In [ ]:
# M2 == "F" - too weak once ranked globally. Within W it looks strong, but the score
# ranks transactions across ALL products within a day, so the pooled figure is the
# decision-relevant one. A rule that finds risky-for-W rows still selects rows less
# risky than an average C transaction, and it drags the ranking.
m = read_train(["day", "isFraud", "TransactionAmt", "ProductCD", "M2"])
m2_f = m["M2"].astype("object").eq("F")
within_w = m[m["ProductCD"].astype(str) == "W"]

rows = [
    lift(within_w, within_w["M2"].astype("object").eq("F"), "M2 == F within W"),
    lift(m, m2_f, "M2 == F pooled"),
]
display(show(rows))
display(show([stability(m, m2_f, "M2 == F pooled")]))

In [ ]:
# Missing billing address - a restatement of R2. addr1/addr2 share an identical null
# mask, nearly every null row is product C, and the pooled lift collapses within C.
addr = read_train(["day", "isFraud", "TransactionAmt", "ProductCD", "addr1", "addr2"])
null_addr = addr["addr1"].isna()

print(f"addr1 and addr2 share a null mask: {null_addr.equals(addr['addr2'].isna())}")
print(
    f"share of addr-null rows that are product C: {addr.loc[null_addr, 'ProductCD'].astype(str).eq('C').mean():.1%}"
)

within_c = addr[addr["ProductCD"].astype(str) == "C"]
show(
    [
        lift(addr, null_addr, "addr null, pooled"),
        lift(within_c, within_c["addr1"].isna(), "addr null within C"),
    ]
)

In [ ]:
# Unusual hour - a denominator artifact, and the most tempting number in the Phase 01
# notebook. The peak bucket's high RATE comes with almost no fraud to catch: legitimate
# volume collapses overnight harder than fraud does, so at a ~35/day review budget the
# rule spends its weight on an empty window.
#
# `hour` is also cyclic and not a wall-clock label - bucket 0 is not midnight - so no
# time-of-day rationale could have been stated honestly in any case.
h = read_train(["day", "isFraud", "TransactionAmt", "hour"])
days = h["day"].nunique()

by_hour = h.groupby("hour").apply(
    lambda g: pd.Series(
        {
            "rows_per_day": round(len(g) / days, 1),
            "frauds_per_day": round((g["isFraud"] == 1).sum() / days, 1),
            "fraud_rate": g["isFraud"].mean(),
            "lift": g["isFraud"].mean() / h["isFraud"].mean(),
        }
    ),
    include_groups=False,
)
by_hour.round(3).sort_values("fraud_rate", ascending=False).head(6)

### Section 8 — the weights, against what is committed

The check this notebook exists for. Every weight in `config/config.yaml` is
`round(min(lift_half_A, lift_half_B))`; this recomputes them and compares.

A mismatch is a **finding**, not a prompt to edit config. The committed weights are
what produced `reports/metrics/rules_baseline.json`; changing them silently would
leave the report describing an engine that no longer exists.

In [ ]:
# `product_tier` is absent: it is a points lookup, not a binary predicate, so it has no
# single lift to round. Its evidence is the Section 3 table.
measured = pd.DataFrame(
    [
        stability(r1, r1_fires, "round_amount"),
        stability(r3, r3_fires, "amount_percentile"),
        stability(r4, r4_fires, "new_card"),
        stability(r5, r5_fires, "w_m4_m2"),
    ]
).set_index("rule")

measured["committed_weight"] = pd.Series(
    {name: rules_cfg[name]["weight"] for name in measured.index}
)
measured["agrees"] = measured["implied_weight"] == measured["committed_weight"]
display(measured.round(3))

if measured["agrees"].all():
    print("\nEvery committed weight reproduces from the half-split rule.")
else:
    disagreeing = measured.index[~measured["agrees"]].tolist()
    print(f"\nDISCREPANCY on {disagreeing} — record it in docs/rules-baseline.md.")
    print("Do NOT edit config.yaml to match: the committed weights are what produced")
    print("reports/metrics/rules_baseline.json.")

### Section 9 — what this notebook does not settle

1. **The `search` rules are still selection-biased.** The half-split check prices that
   bias; it does not remove it. R4 and R5 were the best two of roughly twenty
   comparisons, and their full-train lift is optimistic by an amount nothing here
   measures.

2. **The real-valued-weight comparison is not reproduced.** `rules-baseline.md` records
   that an exploratory run found it scored no better on train. That measurement predates
   the pipeline and no figure for it is quoted, here or there — it licenses no claim
   about which scheme generalises.

3. **The tiebreaker's effect on ambiguity is not measured here.** Whatever
   `ambiguous_days` reports on VAL-FIT and VAL-CAL is the published number, and it is
   reported rather than engineered away.

4. **Two figures quoted in `rules-baseline.md` do not reproduce here, and both are
   cross-references to Phase 01 rather than Phase 03 measurements.**

   - The H2 `ProductCD` control ($300 at 4.74× pooled → 14.58× within H; $450 at 12.94×
     → 24.20×) comes from `hypotheses.md`, whose evidence window is days 1–120. On train
     alone this notebook measures 3.87× → 9.35× and 9.75× → 13.71×. Recomputing on days
     1–120 gives 4.19× → 9.03×, which still does not close the gap — so the difference
     is not only the window, and `hypotheses.md` appears to compute its lift by a
     definition this notebook does not reproduce. **The direction H2 claimed holds
     either way, and it is the direction the rule rests on.** The magnitudes are Phase
     01's to reconcile.
   - `no identity within C` reads 0.57× here against the 0.54× quoted. Same class of
     discrepancy, far smaller, and it does not touch a decision — the rule was rejected
     for being a restatement of R2, which reproduces exactly at 98.6% agreement.

   Neither changes a weight, so nothing in `config.yaml` moves. Recorded because the
   whole point of this notebook is that a number nobody can re-run is not evidence.

5. **R1's band edges were formed on days 1–120**, which includes the purge gap. Not
   leakage — the gap is purged for label maturity, not held out for evaluation, and
   everything evaluated on begins at day 121 — but recorded because it is a fair
   question.